In [ ]:
# =============================================================================
# CELL 1 — Tải dataset 
# =============================================================================
# Chạy cell này 1 lần, mất ~5-10 phút tuỳ tốc độ mạng Kaggle
# Nếu đã tải rồi thì skip

import os
file_path = "/kaggle/working/goodreads_interactions.csv"

if not os.path.exists(file_path):
    print("Bắt đầu tải file dataset (khoảng 4GB)...")
    print("Vui lòng kiên nhẫn đợi 5 - 10 phút. Đang tải ngầm để tránh đơ máy...")
    
    # Đã bỏ cờ --show-progress
    os.system(f"wget -q https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions.csv -O {file_path}")
    
    print("Download xong hoàn tất!")
else:
    print("File đã có, skip download.")

Bắt đầu tải file dataset (khoảng 4GB)...
Vui lòng kiên nhẫn đợi 5 - 10 phút. Đang tải ngầm để tránh đơ máy...
Download xong hoàn tất!


In [ ]:
# =============================================================================
# CELL 2 — Data Loader
# =============================================================================

from __future__ import annotations
import json, pickle, gc
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, asdict
from typing import Optional
import numpy as np
import pandas as pd

@dataclass
class DataConfig:
    raw_csv: str = "/kaggle/working/goodreads_interactions.csv"
    out_dir: str = "/kaggle/working/processed"
    n_stages: int = 4
    recommend_threshold: int = 4
    min_user_inter: int = 5
    min_item_inter: int = 5
    require_final_stage: bool = True
    sample_n_users: Optional[int] = None
    sample_seed: int = 1234
    n_val_users: int = 5000
    n_test_users: int = 5000

class GoodreadsChainLoader:
    def __init__(self, cfg: DataConfig):
        self.cfg = cfg
        Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)
        self.user_idx: dict = {}
        self.item_idx: dict = {}
        self.n_user: int = 0
        self.n_item: int = 0
        self.user_item_map: dict = defaultdict(set)

    def _extract_raw_data(self, chunksize=1_000_000):
        print("[1/5] Extracting raw interactions with Fast Proxy Timestamp...")
        rows, n_rows = [], 0
        usecols = ["user_id", "book_id", "is_read", "rating"]

        global_row = 0
        for chunk in pd.read_csv(self.cfg.raw_csv, usecols=usecols, chunksize=chunksize,
                                 dtype={"user_id":np.int32, "book_id":np.int32,
                                        "is_read":np.int8, "rating":np.int8}):
            stage = np.zeros(len(chunk), dtype=np.int8)
            stage[chunk["is_read"].values == 1] = 1
            rated = chunk["rating"].values > 0
            stage[rated] = np.maximum(stage[rated], 2)
            recom = chunk["rating"].values >= self.cfg.recommend_threshold
            stage[recom] = np.maximum(stage[recom], 3)

            # Sử dụng hàng chỉ mục làm proxy timestamp để tránh parse datetime chậm
            ts = np.arange(global_row, global_row + len(chunk), dtype=np.int32)
            global_row += len(chunk)

            rows.append(np.stack([chunk["user_id"].values, chunk["book_id"].values, stage, ts], axis=1))
            n_rows += len(chunk)
            print(f"    ...{n_rows:,} rows read", end="\r")
            del chunk, stage, rated, recom, ts
            gc.collect()

        raw_data = np.concatenate(rows).astype(np.int32)
        del rows
        gc.collect()
        return raw_data

    def _temporal_split(self, raw_data):
        print("[2/5] Performing Temporal Leave-last-out & User Sampling...")
        df = pd.DataFrame(raw_data, columns=['u', 'i', 's', 'ts'])
        del raw_data
        
        df = df.sort_values(['u', 'ts'], ascending=[True, True])
        
        counts = df.groupby('u').size()
        valid_users = counts[counts >= 3].index
        
       
        if self.cfg.sample_n_users is not None and len(valid_users) > self.cfg.sample_n_users:
            print(f"    -> Sampling {self.cfg.sample_n_users:,} users from {len(valid_users):,} total users...")
            rng = np.random.default_rng(self.cfg.sample_seed)
            valid_users = rng.choice(valid_users, size=self.cfg.sample_n_users, replace=False)
            df = df[df['u'].isin(valid_users)].copy()
        
        last_indices = df.groupby('u').tail(1).index
        potential_test_users = df.loc[last_indices]['u'].values
        
        rng = np.random.default_rng(self.cfg.sample_seed)
        rng.shuffle(potential_test_users)
        
        n_val = self.cfg.n_val_users
        n_test = self.cfg.n_test_users
        val_u = set(potential_test_users[:n_val])
        test_u = set(potential_test_users[n_val : n_val + n_test])
        
        is_val = df.index.isin(last_indices) & df['u'].isin(val_u)
        is_test = df.index.isin(last_indices) & df['u'].isin(test_u)
        
        self.df_val = df[is_val].drop(columns=['ts'])
        self.df_test = df[is_test].drop(columns=['ts'])
        self.df_train = df[~(is_val | is_test)].drop(columns=['ts'])
        
        print(f"    Split sizes -> Train: {len(self.df_train):,}, Val: {len(self.df_val):,}, Test: {len(self.df_test):,}")

    def _train_only_filter_and_map(self):
        print("[3/5] K-core Filtering (Train set only)...")
        df_tr = self.df_train
        iteration = 1
        while True:
            u_counts = df_tr['u'].value_counts()
            i_counts = df_tr['i'].value_counts()
            
            valid_u = u_counts[u_counts >= self.cfg.min_user_inter].index
            valid_i = i_counts[i_counts >= self.cfg.min_item_inter].index
            
            if len(valid_u) == len(u_counts) and len(valid_i) == len(i_counts):
                break
                
            df_tr = df_tr[df_tr['u'].isin(valid_u) & df_tr['i'].isin(valid_i)]
            print(f"    ...iter {iteration}: {len(df_tr):,} interactions left")
            iteration += 1
            if iteration > 5: break
                
        valid_u_set = set(df_tr['u'].unique())
        valid_i_set = set(df_tr['i'].unique())
        print(f"    Train core-filtered: {len(valid_u_set):,} users, {len(valid_i_set):,} items.")
        
        print("[4/5] Aligning Val/Test (Cold-start Drop) & Re-indexing...")
        self.df_val  = self.df_val[self.df_val['u'].isin(valid_u_set) & self.df_val['i'].isin(valid_i_set)]
        self.df_test = self.df_test[self.df_test['u'].isin(valid_u_set) & self.df_test['i'].isin(valid_i_set)]
        
        self.user_idx = {u: i for i, u in enumerate(sorted(valid_u_set))}
        self.item_idx = {item: i for i, item in enumerate(sorted(valid_i_set))}
        self.n_user = len(self.user_idx)
        self.n_item = len(self.item_idx)
        
        self.data_train = df_tr.copy()
        self.data_val   = self.df_val.copy()
        self.data_test  = self.df_test.copy()
        
        for d in [self.data_train, self.data_val, self.data_test]:
            d['u'] = d['u'].map(self.user_idx)
            d['i'] = d['i'].map(self.item_idx)
            
        self.data_train = self.data_train.values.astype(np.int32)
        self.data_val   = self.data_val.values.astype(np.int32)
        self.data_test  = self.data_test.values.astype(np.int32)
        
        del self.df_train, self.df_val, self.df_test, df_tr
        gc.collect()
        print(f"    Final sizes -> Train: {len(self.data_train):,}, Val: {len(self.data_val):,}, Test: {len(self.data_test):,}")

    def _build_train_user_item_map(self):
        print("[5/5] Building user-item map (STRICTLY from Train)...")
        self.user_item_map = defaultdict(set)
        for u, i, _ in self.data_train:
            self.user_item_map[int(u)].add(int(i))
            
        stage_counts = np.bincount(self.data_train[:, 2], minlength=self.cfg.n_stages)
        print("    Train stage dist: " + ", ".join(f"stage{l}={c:,}" for l, c in enumerate(stage_counts)))

    def build(self):
        raw_data = self._extract_raw_data()
        self._temporal_split(raw_data)
        self._train_only_filter_and_map()
        self._build_train_user_item_map()
        return self

    def save(self):
        out = Path(self.cfg.out_dir)
        np.save(out/"data_train.npy", self.data_train)
        np.save(out/"data_val.npy",   self.data_val)
        np.save(out/"data_test.npy",  self.data_test)
        pickle.dump(self.user_idx,        open(out/"user_idx.pkl","wb"))
        pickle.dump(self.item_idx,        open(out/"item_idx.pkl","wb"))
        pickle.dump(dict(self.user_item_map), open(out/"user_item_map.pkl","wb"))
        json.dump({"n_user":self.n_user,"n_item":self.n_item,"n_stage":self.cfg.n_stages,
                   "config":asdict(self.cfg)},
                  open(out/"meta.json","w"), indent=2)
        print(f"Saved to {out}/")

In [ ]:
# =============================================================================
# CELL 3 — Khởi tạo cấu hình và Tiến hành xử lý dữ liệu (MỚI)
# =============================================================================
import shutil

# Khởi tạo cấu hình - Đặt mẫu 30,000 users để dung lượng data_train rơi vào khoảng ~4-5 triệu dòng
cfg_data = DataConfig(
    raw_csv="/kaggle/working/goodreads_interactions.csv",
    out_dir="/kaggle/working/processed",
    sample_n_users=30000, 
    n_val_users=5000,
    n_test_users=5000
)


old_proc = Path(cfg_data.out_dir)
if old_proc.exists():
    shutil.rmtree(old_proc)

loader = GoodreadsChainLoader(cfg_data)
loader.build()
loader.save()

[1/5] Extracting raw interactions with Fast Proxy Timestamp...
[2/5] Performing Temporal Leave-last-out & User Sampling...
    -> Sampling 30,000 users from 833,843 total users...
    Split sizes -> Train: 8,210,266, Val: 5,000, Test: 5,000
[3/5] K-core Filtering (Train set only)...
    ...iter 1: 6,953,693 interactions left
    ...iter 2: 6,952,403 interactions left
    ...iter 3: 6,952,131 interactions left
    Train core-filtered: 28,702 users, 238,562 items.
[4/5] Aligning Val/Test (Cold-start Drop) & Re-indexing...
    Final sizes -> Train: 6,952,131, Val: 3,671, Test: 3,678
[5/5] Building user-item map (STRICTLY from Train)...
    Train stage dist: stage0=3,508,332, stage1=219,495, stage2=947,867, stage3=2,276,437
Saved to /kaggle/working/processed/


In [4]:
# =============================================================================
# CELL 4 — Load processed data
# =============================================================================

from pathlib import Path
import numpy as np
import json

PROC = Path("/kaggle/working/processed")

# Kiểm tra file tồn tại trước khi load
required_files = ["data_train.npy", "data_val.npy", "data_test.npy", "meta.json", "user_item_map.pkl"]
missing = [f for f in required_files if not (PROC / f).exists()]

if missing:
    raise FileNotFoundError(
        f"Thiếu các file sau: {missing}\n"
        f"→ Hãy chạy lại Cell 3 (data loader) trước, đảm bảo không có lỗi."
    )

data_train = np.load(PROC / "data_train.npy")
data_val   = np.load(PROC / "data_val.npy")
data_test  = np.load(PROC / "data_test.npy")

with open(PROC / "meta.json") as f:
    meta = json.load(f)

import pickle
with open(PROC / "user_item_map.pkl", "rb") as f:
    user_item_map = pickle.load(f)

print("✓ Load xong!")
print(f"  data_train : {data_train.shape}")
print(f"  data_val   : {data_val.shape}")
print(f"  data_test  : {data_test.shape}")
print(f"  n_user={meta['n_user']}, n_item={meta['n_item']}, n_stage={meta['n_stage']}")


✓ Load xong!
  data_train : (6952131, 3)
  data_val   : (3671, 3)
  data_test  : (3678, 3)
  n_user=28702, n_item=238562, n_stage=4


In [ ]:
# =============================================================================
# CELL 5 — chainRec Model (Phase 1) 
# =============================================================================

import time
from dataclasses import dataclass
from typing import Literal
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

@dataclass
class ModelConfig:
    n_user: int
    n_item: int
    n_stage: int       = 4
    embed_dim: int     = 64
    beta: float        = 1.0
    learn_beta: bool   = True
    l2: float          = 0.01
    lr: float          = 0.001
    batch_size: int    = 512
    n_neg: int         = 1
    n_epochs: int      = 50
    patience: int      = 5
    sampler: Literal["uniform", "stagewise"] = "uniform"
    device: str        = "cuda" if torch.cuda.is_available() else "cpu"

class ChainRecModel(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        K, L = cfg.embed_dim, cfg.n_stage
        self.user_emb  = nn.Embedding(cfg.n_user, K)
        self.item_emb  = nn.Embedding(cfg.n_item, K)
        self.stage_emb = nn.Embedding(L, K)
        self.b0        = nn.Parameter(torch.zeros(1))
        self.b_user    = nn.Embedding(cfg.n_user, 1)
        self.b_item    = nn.Embedding(cfg.n_item, 1)
        log_beta_init  = torch.log(torch.tensor(float(cfg.beta)))
        if cfg.learn_beta:
            self.log_beta = nn.Parameter(log_beta_init)
        else:
            self.register_buffer("log_beta", log_beta_init)
        for emb in [self.user_emb, self.item_emb, self.stage_emb]:
            nn.init.xavier_uniform_(emb.weight)
        for bias in [self.b_user, self.b_item]:
            nn.init.zeros_(bias.weight)

    @property
    def beta(self):
        return torch.clamp(self.log_beta.exp(), min=1.0)

    def _intention_score(self, u, i, l):
        return (self.stage_emb(l) * self.item_emb(i) * self.user_emb(u)).sum(-1)

    def _rectified(self, delta):
        b = self.beta
        return F.softplus(b * delta) / b

    def score(self, u, i, target_stage):
        B = u.shape[0]
        bias = self.b0 + self.b_user(u).squeeze(-1) + self.b_item(i).squeeze(-1)
        acc = torch.zeros(B, device=u.device)
        for lp in range(target_stage, self.cfg.n_stage):
            l_t = torch.full((B,), lp, dtype=torch.long, device=u.device)
            acc = acc + self._rectified(self._intention_score(u, i, l_t))
        return bias + acc

    def score_all_stages(self, u, i):
        B = u.shape[0]
        bias = self.b0 + self.b_user(u).squeeze(-1) + self.b_item(i).squeeze(-1)
        delta_plus = torch.stack([
            self._rectified(self._intention_score(
                u, i, torch.full((B,), l, dtype=torch.long, device=u.device)))
            for l in range(self.cfg.n_stage)
        ], dim=1)
        suffix_sum = delta_plus.flip(dims=[1]).cumsum(dim=1).flip(dims=[1])
        return bias.unsqueeze(1) + suffix_sum

    def edgewise_terms(self, u, i, l_star):
        L = self.cfg.n_stage
        scores = self.score_all_stages(u, i)
        l_star_c = l_star.clamp(0, L-1)
        s_lstar  = scores.gather(1, l_star_c.unsqueeze(1)).squeeze(1)
        s_next   = scores.gather(1, (l_star+1).clamp(0,L-1).unsqueeze(1)).squeeze(1)
        s_next   = torch.where(l_star == L-1, torch.full_like(s_next, -1e9), s_next)
        p_lstar  = torch.sigmoid(s_lstar)
        p_next   = torch.sigmoid(s_next)
        delta_p  = self._rectified(self._intention_score(u, i, l_star_c))
        p_cap    = (1.0 - torch.exp(-delta_p)).clamp(min=1e-8)
        return p_lstar, p_next, p_cap

def edgewise_loss(model, u_pos, i_pos, l_pos, u_neg, i_neg, l_neg, l2):
    p_pos, _, _       = model.edgewise_terms(u_pos, i_pos, l_pos)
    _, p_next, p_cap  = model.edgewise_terms(u_neg, i_neg, l_neg)
    loss_pos = -torch.log(p_pos.clamp(min=1e-8)).mean()
    loss_neg = -(torch.log((1-p_next).clamp(min=1e-8)) + torch.log(p_cap)).mean()
    l2_loss  = l2 * (model.user_emb.weight.norm(2)**2 + model.item_emb.weight.norm(2)**2) \
               / (model.cfg.n_user + model.cfg.n_item)
    return loss_pos + loss_neg + l2_loss

class ChainDataset(Dataset):
    def __init__(self, data, user_item_map, n_item, n_neg=1,
                 sampler="uniform", all_data=None):
        self.data = data
        self.user_item_map = user_item_map
        self.n_item = n_item
        self.n_neg  = n_neg
        self.sampler = sampler
        self.rng = np.random.default_rng(42)
        if sampler == "stagewise" and all_data is not None:
            self.user_stage_count = defaultdict(lambda: defaultdict(int))
            for u,i,l in all_data:
                self.user_stage_count[int(u)][int(l)] += 1
        else:
            self.user_stage_count = None

    def _sample_neg(self, u, l_star):
        pos = self.user_item_map.get(u, set())
        for _ in range(30):
            ni = self.rng.integers(0, self.n_item)
            if ni not in pos:
                return int(ni), int(l_star)
        return int(self.rng.integers(0, self.n_item)), int(l_star)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        u, i, l = [int(x) for x in self.data[idx]]
        ni, nl  = self._sample_neg(u, l)
        return {
            "u_pos": torch.tensor(u,  dtype=torch.long),
            "i_pos": torch.tensor(i,  dtype=torch.long),
            "l_pos": torch.tensor(l,  dtype=torch.long),
            "u_neg": torch.tensor(u,  dtype=torch.long),
            "i_neg": torch.tensor(ni, dtype=torch.long),
            "l_neg": torch.tensor(nl, dtype=torch.long),
        }

class Evaluator:
    def __init__(self, model, data_test, user_item_map, n_item,
                 n_neg_eval=500, K_list=None, device="cpu"):
        self.model        = model
        self.data_test    = data_test
        self.user_item_map= user_item_map
        self.n_item       = n_item
        self.n_neg_eval   = n_neg_eval
        self.K_list       = K_list or [10, 20]
        self.device       = device
        self.rng          = np.random.default_rng(999)

    @torch.no_grad()
    def evaluate(self, target_stage=None):
        self.model.eval()
        L = self.model.cfg.n_stage
        if target_stage is None: target_stage = L - 1
        hits = {k: [] for k in self.K_list}
        ndcg = {k: [] for k in self.K_list}
        aucs = []
        for u, i_pos, l_star in self.data_test:
            u, i_pos = int(u), int(i_pos)
            if int(l_star) != target_stage: continue
            pos_items = self.user_item_map.get(u, set())
            negs, attempts = [], 0
            while len(negs) < self.n_neg_eval and attempts < self.n_neg_eval * 5:
                c = self.rng.integers(0, self.n_item)
                if c not in pos_items and c != i_pos: negs.append(c)
                attempts += 1
            all_items = np.array([i_pos] + negs, dtype=np.int64)
            u_t = torch.full((len(all_items),), u, dtype=torch.long, device=self.device)
            i_t = torch.tensor(all_items, dtype=torch.long, device=self.device)
            scores   = self.model.score(u_t, i_t, target_stage).cpu().numpy()
            ranked   = np.argsort(-scores)
            pos_rank = int(np.where(ranked == 0)[0][0])
            aucs.append((scores[0] > scores[1:]).mean())
            for k in self.K_list:
                hit = int(pos_rank < k)
                hits[k].append(hit)
                ndcg[k].append((hit / np.log2(pos_rank+2)) / (1/np.log2(2)) if hit else 0.0)
        results = {"AUC": float(np.mean(aucs))}
        for k in self.K_list:
            results[f"Recall@{k}"] = float(np.mean(hits[k]))
            results[f"NDCG@{k}"]   = float(np.mean(ndcg[k]))
        return results

class Trainer:
    def __init__(self, model, cfg, data_train, data_val, data_test, user_item_map):
        self.model     = model.to(cfg.device)
        self.cfg       = cfg
        self.device    = cfg.device
        self.data_val  = data_val
        self.optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
        ds = ChainDataset(data_train, user_item_map, cfg.n_item,
                          n_neg=cfg.n_neg, sampler=cfg.sampler, all_data=data_train)
        
        # SỬA TRIỆT ĐỂ LỖI MULTIPROCESSING: set num_workers=0
        self.train_dl  = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True,
                                    num_workers=0, pin_memory=(cfg.device=="cuda"))
        
        self.evaluator = Evaluator(model, data_test, user_item_map,
                                   cfg.n_item, device=cfg.device)

    def _val_loss(self):
        self.model.eval()
        with torch.no_grad():
            idx   = np.random.permutation(len(self.data_val))[:2000]
            batch = self.data_val[idx]
            u = torch.tensor(batch[:,0], dtype=torch.long, device=self.device)
            i = torch.tensor(batch[:,1], dtype=torch.long, device=self.device)
            l = torch.tensor(batch[:,2], dtype=torch.long, device=self.device)
            p, _, _ = self.model.edgewise_terms(u, i, l)
            return -torch.log(p.clamp(min=1e-8)).mean().item()

    def train(self, save_path="/kaggle/working/chainrec_best.pt"):
        best_val, patience_count, history = float("inf"), 0, []
        print(f"Training on {self.device}")
        print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | "
              f"{'AUC':>6} | {'R@10':>6} | {'NDCG@10':>7} | {'Time':>6}")
        print("-" * 65)
        for epoch in range(1, self.cfg.n_epochs + 1):
            self.model.train()
            t0, total, nb = time.time(), 0.0, 0
            for batch in self.train_dl:
                u_pos = batch["u_pos"].to(self.device)
                i_pos = batch["i_pos"].to(self.device)
                l_pos = batch["l_pos"].to(self.device)
                u_neg = batch["u_neg"].to(self.device)
                i_neg = batch["i_neg"].to(self.device)
                l_neg = batch["l_neg"].to(self.device)
                self.optimizer.zero_grad()
                loss = edgewise_loss(self.model, u_pos, i_pos, l_pos,
                                     u_neg, i_neg, l_neg, self.cfg.l2)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                total += loss.item(); nb += 1
            train_loss = total / nb
            val_loss   = self._val_loss()
            metrics    = self.evaluator.evaluate()
            print(f"{epoch:>5} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
                  f"{metrics['AUC']:>6.4f} | {metrics['Recall@10']:>6.4f} | "
                  f"{metrics['NDCG@10']:>7.4f} | {time.time()-t0:>5.1f}s")
            history.append({"epoch":epoch,"train_loss":train_loss,
                            "val_loss":val_loss,**metrics})
            if val_loss < best_val:
                best_val, patience_count = val_loss, 0
                torch.save(self.model.state_dict(), save_path)
            else:
                patience_count += 1
                if patience_count >= self.cfg.patience:
                    print(f"Early stopping at epoch {epoch}")
                    break
        self.model.load_state_dict(torch.load(save_path, map_location=self.device))
        print("\n=== Final Test Results ===")
        final = self.evaluator.evaluate()
        for k,v in final.items(): print(f"  {k}: {v:.4f}")
        return history, final

In [6]:
# =============================================================================
# CELL 6 — Train 
# =============================================================================

import torch

cfg = ModelConfig(
    n_user     = meta["n_user"],
    n_item     = meta["n_item"],
    n_stage    = meta["n_stage"],
    embed_dim  = 16,     
    l2         = 0.01,
    lr         = 0.001,
    batch_size = 1024,   
    n_neg      = 1,
    n_epochs   = 50,
    patience   = 5,
    sampler    = "uniform",
)

model = ChainRecModel(cfg)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {cfg.device}")

trainer = Trainer(model, cfg, data_train, data_val, data_test, user_item_map)
history, final_metrics = trainer.train()

Parameters: 4,543,554
Device: cuda
Training on cuda
Epoch | Train Loss |   Val Loss |    AUC |   R@10 | NDCG@10 |   Time
-----------------------------------------------------------------
    1 |     1.0777 |     0.3292 | 0.8851 | 0.5023 |  0.3353 | 426.1s
    2 |     0.5330 |     0.2832 | 0.9028 | 0.5152 |  0.3484 | 423.6s
    3 |     0.4567 |     0.2935 | 0.9099 | 0.5269 |  0.3542 | 423.8s
    4 |     0.4152 |     0.3079 | 0.9139 | 0.5351 |  0.3516 | 424.9s
    5 |     0.3836 |     0.3074 | 0.9183 | 0.5433 |  0.3642 | 424.4s
    6 |     0.3586 |     0.3427 | 0.9222 | 0.5515 |  0.3663 | 425.5s
    7 |     0.3382 |     0.3358 | 0.9252 | 0.5468 |  0.3665 | 425.4s
Early stopping at epoch 7

=== Final Test Results ===
  AUC: 0.9024
  Recall@10: 0.5258
  NDCG@10: 0.3463
  Recall@20: 0.6230
  NDCG@20: 0.3709


In [7]:

# =============================================================================
# CELL 7 — Multi-stage evaluation
# =============================================================================

stage_names = ["shelve", "read", "rate", "recommend"]
print("\n=== Per-Stage Evaluation ===")
for stage in range(meta["n_stage"]):
    stage_data = data_test[data_test[:, 2] == stage]
    if len(stage_data) == 0:
        continue
    ev = Evaluator(model, stage_data, user_item_map,
                   meta["n_item"], device=cfg.device)
    m  = ev.evaluate(target_stage=stage)
    print(f"Stage {stage} ({stage_names[stage]:>10}): "
          f"AUC={m['AUC']:.4f}, R@10={m['Recall@10']:.4f}, NDCG@10={m['NDCG@10']:.4f}")


=== Per-Stage Evaluation ===
Stage 0 (    shelve): AUC=0.8178, R@10=0.3472, NDCG@10=0.2115
Stage 1 (      read): AUC=0.8656, R@10=0.4081, NDCG@10=0.2573
Stage 2 (      rate): AUC=0.8541, R@10=0.4343, NDCG@10=0.3109
Stage 3 ( recommend): AUC=0.9034, R@10=0.5246, NDCG@10=0.3460
